In [ ]:
import numpy as np
import pandas as pd
from Bio import SeqIO
import os

#============================================================
# 路径配置
# ============================================================
# detail_csv = "/home/nick/data/prokaryo/ADI_predict_prepare_data/new_data/align_data/ref_usalign_1rxx/token_labels_detail_multi.csv"
detail_csv = "/home/nick/data/prokaryo/ADI_predict_prepare_data/new_data/align_data/ref_usalign_1rxx/token_labels_detail_multi_labels.csv"
fasta_file = '/home/nick/data/software/scripts/aminodegrade_20260109/ADI/uniprotKB/agument_pos/uniprotkb_ec_3_5_3_6_pos_aug.fasta'
output_dir = "/home/nick/data/prokaryo/ADI_predict_prepare_data/new_data/align_data/ref_usalign_1rxx/npy_labels_new_multi_labels/"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# 标签个数（分类数）设置，假设最大标签编号是19（含0）
# ============================================================
num_labels = 5

# ============================================================
# 辅助函数：解析多标签字符串为标签列表
# ============================================================
def parse_multi_label(label_str):
    """
    把类似 '2-4' 解析为标签列表[2,4]
    0或'0'返回[0]，'-1'代表masked返回空列表
    """
    if label_str in ['-1', -1]:
        return [0]  # Masked 无标签
    if label_str == '0' or label_str == 0:
        return [0]  # 背景标签0
    parts = str(label_str).split('-')
    labels = []
    for p in parts:
        try:
            labels.append(int(p))
        except:
            pass
    return labels

def encode_multi_hot(label_list, num_labels):
    """
    把标签列表编码成multi-hot向量
    0表示背景，出现时background=1，其它标签出现时background=0
    """
    vec = np.zeros(num_labels, dtype=np.uint8)
    if 0 in label_list:
        # 只有背景标签时，标记背景为1
        vec[0] = 1
    else:
        # 有其它标签，则对应位置1，背景标0
        for l in label_list:
            if 0 <= l < num_labels:
                vec[l] = 1
        vec[0] = 0
    return vec

def decode_multi_hot_vector(vec):
    """
    把multi-hot向量解码为字符串
    例如array([0,1,0,1,...]) -> "1-3"
    若背景位1，则返回"0"
    """
    if vec[0] == 1 and vec.sum() == 1:
        return "0"
    label_indices = [str(i) for i, val in enumerate(vec) if val == 1]
    return '-'.join(label_indices)

# ============================================================
# 1. 读取 detail 表
# ============================================================
full_df = pd.read_csv(detail_csv)

print(f"Detail CSV 行数: {len(full_df)}")
print(f"标签示例: {full_df['label'].unique()[:10]}")

# ============================================================
# 2. 读取 FASTA 序列，建立名称到序列的映射
# ============================================================
sequences = {}
for record in SeqIO.parse(fasta_file, "fasta"):
    key = record.id
    sequences[key] = str(record.seq)
    # 也存取无后缀的版本方便匹配
    short_key = key.replace("-F1-model_v6", "").replace("-F1-model", "")
    if short_key != key:
        sequences[short_key.replace('AF-', '')] = str(record.seq)

print(f"FASTA 中共 {len(sequences)} 条序列")

# ============================================================
# 3. 匹配 + 生成多标签矩阵 + 保存
# ============================================================

matched = 0
unmatched_csv = []
all_labels_dict = {}

unique_files = full_df['file'].unique()

for file_name in unique_files:
    # if file_name=='AF-A0A010RP19-F1-model_v6':
        seq = None
        matched_key = None
        for candidate in [file_name,
                          file_name.replace("-F1-model_v6", ""),
                          file_name.replace("-F1-model", ""),
                          "AF-" + file_name]:
            if candidate.startswith('AF-'):
                candidate = candidate.replace('AF-', '')
            if candidate in sequences:
                seq = sequences[candidate]
                matched_key = candidate
                break
    
        if seq is None:
            unmatched_csv.append(file_name)
            continue
    
        sub_df = full_df[full_df['file'] == file_name]
        seq_len = len(seq)
        # print(sub_df)
        # 初始化 多标签矩阵 shape = (seq_len, num_labels)
        labels = np.zeros((seq_len, num_labels), dtype=np.uint8)
        label_str_list=[]
        for _, row in sub_df.iterrows():
            pos = int(row['query_pos'])
            label_str = row['label']
            label_str_list.append(label_str)
            # if label_str == '2-4':
                # print(label_str)
                # break
            if 1 <= pos <= seq_len:
                label_list = parse_multi_label(label_str)
                multi_hot_vec = encode_multi_hot(label_list, num_labels)
                labels[pos - 1] = multi_hot_vec
    
        file_name_fixed = file_name.replace("-F1-model_v6", "").replace("AF-", "")
        npy_path = os.path.join(output_dir, f"{file_name_fixed}.npy")
        np.save(npy_path, labels)
    
        all_labels_dict[file_name_fixed] = {
            'sequence': seq,
            'labels': labels
        }
        matched += 1

print(f"\n匹配成功: {matched}")
if unmatched_csv:
    print(f"未匹配: {len(unmatched_csv)} 条文件，示例:")
    for name in unmatched_csv[:5]:
        print(f"  {name}")
print(label_str_list)
